# Olist Brazilian E-Commerce — Figure NotebookGenerates every figure for the LaTeX report, straight from BigQuery, using **SciencePlots**.**Story spine:** São Paulo supply concentration → long shipping distance → slow/costly delivery→ poor reviews → 3.1% repeat rate.Each figure is saved to `figures/` as **PDF** (vector, for LaTeX) and **PNG** (for quick viewing).Drop the PDFs straight into your document with `\includegraphics{figures/fig_XX_name.pdf}`.---### How to use1. Run **Setup** (cells 1–4) once per session.2. Run any chapter section independently — each is self-contained.3. Figures land in `figures/`; the last cell zips them for download.

## 1 · Setup

In [ ]:
# Install SciencePlots. Add the LaTeX toolchain only for FINAL export# (it takes a few minutes; the 'no-latex' style is fine while iterating).!pip install SciencePlots --quietUSE_LATEX = False   # flip to True for final report-quality typographyif USE_LATEX:    !apt-get install -y texlive-latex-extra texlive-fonts-recommended dvipng cm-super -qq

In [ ]:
# Authenticate to BigQueryfrom google.colab import authauth.authenticate_user()print("Authenticated.")

In [ ]:
import warnings, oswarnings.filterwarnings("ignore")import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport scienceplotsfrom google.cloud import bigquery# ---- CONFIG ---------------------------------------------------------------PROJECT = "blinkit-analysis-503505"   # <-- change if your project differsDATASET = "olist"FIGDIR  = "figures"# ---------------------------------------------------------------------------os.makedirs(FIGDIR, exist_ok=True)client = bigquery.Client(project=PROJECT)STYLE = ["science", "grid"] if USE_LATEX else ["science", "no-latex", "grid"]plt.style.use(STYLE)plt.rcParams.update({    "figure.dpi": 120,    "savefig.dpi": 300,    "font.size": 9,    "axes.titlesize": 10,    "axes.labelsize": 9,    "legend.fontsize": 8,})# Consistent palette across every figureC_MAIN, C_ACCENT, C_WARN, C_MUTED = "#0C5DA5", "#00B945", "#FF2C00", "#845B97"print("Ready. Style:", STYLE)

In [ ]:
def q(sql: str) -> pd.DataFrame:    """Run a query against the project and return a DataFrame."""    return client.query(sql.replace("`olist.", f"`{PROJECT}.{DATASET}.")).to_dataframe()def save(fig, name: str):    """Save a figure as PDF (LaTeX) + PNG (preview)."""    fig.savefig(f"{FIGDIR}/{name}.pdf", bbox_inches="tight")    fig.savefig(f"{FIGDIR}/{name}.png", bbox_inches="tight")    print("saved:", name)def barlabels(ax, bars, fmt="{:.0f}", dy=0.01):    """Write value labels above bars."""    top = ax.get_ylim()[1]    for b in bars:        ax.text(b.get_x() + b.get_width()/2, b.get_height() + top*dy,                fmt.format(b.get_height()), ha="center", va="bottom", fontsize=7)# Smoke test — confirms auth + dataset are reachable_t = q("SELECT COUNT(*) AS n FROM `olist.v_orders_enriched`")print("v_orders_enriched rows:", int(_t.n[0]))

---## 2 · Chapters 1–2 — Business overview & temporal patterns*Establishes the terrain: how big Olist is, how fast it grew, and when Brazil shops.*

In [ ]:
# Fig 01 — Monthly order growthdf = q("""SELECT DATE_TRUNC(DATE(purchase_ts), MONTH) AS order_month,       COUNT(*) AS orders,       COUNT(DISTINCT customer_unique_id) AS active_customersFROM `olist.v_orders_enriched`GROUP BY order_month ORDER BY order_month""")df["order_month"] = pd.to_datetime(df.order_month)d = df[(df.order_month >= "2017-01-01") & (df.order_month <= "2018-08-31")]fig, ax = plt.subplots(figsize=(6.5, 3.2))ax.plot(d.order_month, d.orders, marker="o", ms=3.5, color=C_MAIN, label="Orders")bf = d[d.order_month == "2017-11-01"]if len(bf):    ax.annotate("Black Friday", xy=(bf.order_month.iloc[0], bf.orders.iloc[0]),                xytext=(-58, 14), textcoords="offset points", fontsize=8, color=C_WARN,                arrowprops=dict(arrowstyle="->", color=C_WARN, lw=0.8))ax.set_xlabel("Month"); ax.set_ylabel("Orders")ax.set_title("Order volume grew sharply through 2017, then plateaued")ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))fig.autofmt_xdate()save(fig, "fig_01_monthly_growth"); plt.show()

In [ ]:
# Fig 02 — Order status funneldf = q("""SELECT order_status, COUNT(*) AS orders,       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pctFROM `olist.v_orders_enriched` GROUP BY order_status ORDER BY orders DESC""")fig, ax = plt.subplots(figsize=(6, 3))colors = [C_MAIN if s == "delivered" else C_WARN for s in df.order_status]b = ax.barh(df.order_status[::-1], df.orders[::-1], color=colors[::-1], height=0.65)for bar, p in zip(b, df.pct[::-1]):    ax.text(bar.get_width()*1.02, bar.get_y()+bar.get_height()/2,            f"{p:.2f}%", va="center", fontsize=7)ax.set_xscale("log")ax.set_xlabel("Orders (log scale)")ax.set_title("Nearly all orders deliver; a small tail leaks to cancelled/unavailable")save(fig, "fig_02_status_funnel"); plt.show()

In [ ]:
# Fig 03 — Top order days (spikes reveal themselves)df = q("""SELECT DATE(purchase_ts) AS order_date, COUNT(*) AS ordersFROM `olist.v_orders_enriched` GROUP BY order_date ORDER BY orders DESC LIMIT 15""")df["order_date"] = pd.to_datetime(df.order_date)df = df.sort_values("orders")fig, ax = plt.subplots(figsize=(6, 3.6))lbl = df.order_date.dt.strftime("%d %b %Y")cols = [C_WARN if (d.month == 11 and d.year == 2017) else C_MAIN for d in df.order_date]ax.barh(lbl, df.orders, color=cols, height=0.68)ax.set_xlabel("Orders"); ax.set_title("The 15 busiest days — Black Friday 2017 dominates (red)")save(fig, "fig_03_top_days"); plt.show()

In [ ]:
# Fig 04 — Monthly volume vs trailing baseline (quantified lift)df = q("""WITH monthly AS (  SELECT DATE_TRUNC(DATE(purchase_ts), MONTH) AS m, COUNT(*) AS orders  FROM `olist.v_orders_enriched`  WHERE DATE(purchase_ts) BETWEEN '2017-01-01' AND '2018-08-31'  GROUP BY m)SELECT m AS order_month, orders,  ROUND(100.0*(orders - AVG(orders) OVER (ORDER BY m ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING))        / NULLIF(AVG(orders) OVER (ORDER BY m ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING),0),1) AS pct_vs_baselineFROM monthly ORDER BY m""")df["order_month"] = pd.to_datetime(df.order_month)d = df.dropna(subset=["pct_vs_baseline"])fig, ax = plt.subplots(figsize=(6.5, 3))cols = [C_ACCENT if v > 0 else C_WARN for v in d.pct_vs_baseline]ax.bar(d.order_month, d.pct_vs_baseline, width=20, color=cols)ax.axhline(0, color="black", lw=0.7)ax.set_ylabel("% vs trailing 3-month average"); ax.set_xlabel("Month")ax.set_title("Demand lift against baseline — festival months stand out")fig.autofmt_xdate()save(fig, "fig_04_festival_lift"); plt.show()

In [ ]:
# Fig 05 — Day-of-week and hour-of-day (side by side)dow = q("""SELECT EXTRACT(DAYOFWEEK FROM purchase_ts) AS dow,       FORMAT_DATE('%a', DATE(purchase_ts)) AS weekday, COUNT(*) AS ordersFROM `olist.v_orders_enriched` GROUP BY dow, weekday ORDER BY dow""")hod = q("""SELECT EXTRACT(HOUR FROM purchase_ts) AS hour, COUNT(*) AS ordersFROM `olist.v_orders_enriched` GROUP BY hour ORDER BY hour""")fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))axes[0].bar(dow.weekday, dow.orders, color=C_MAIN)axes[0].set_title("By day of week"); axes[0].set_ylabel("Orders")axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))axes[1].plot(hod.hour, hod.orders, marker="o", ms=3, color=C_MAIN)axes[1].fill_between(hod.hour, hod.orders, alpha=0.15, color=C_MAIN)axes[1].set_title("By hour of day"); axes[1].set_xlabel("Hour")axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))fig.suptitle("Weekday-heavy, peaking mid-morning and again after work", y=1.04)save(fig, "fig_05_dow_hour"); plt.show()

In [ ]:
# Fig 06 — Day x Hour heatmapdf = q("""SELECT EXTRACT(DAYOFWEEK FROM purchase_ts) AS dow,       FORMAT_DATE('%a', DATE(purchase_ts)) AS weekday,       EXTRACT(HOUR FROM purchase_ts) AS hour, COUNT(*) AS ordersFROM `olist.v_orders_enriched` GROUP BY dow, weekday, hour""")piv = df.pivot_table(index=["dow","weekday"], columns="hour", values="orders", fill_value=0)piv = piv.sort_index(level="dow")labels = [w for _, w in piv.index]fig, ax = plt.subplots(figsize=(7, 2.8))im = ax.imshow(piv.values, aspect="auto", cmap="YlGnBu")ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))ax.set_xlabel("Hour of day")ax.set_title("When Brazil shops: order density by weekday and hour")ax.grid(False)fig.colorbar(im, ax=ax, label="Orders", fraction=0.03, pad=0.02)save(fig, "fig_06_heatmap_day_hour"); plt.show()

---## 3 · Chapter 3 — Geography: the supply/demand imbalance*The structural root of the story. Verified: sellers are 59.7% São Paulo, customers 42.0% —so 58% of demand sits outside SP while only 40% of supply does.*

In [ ]:
# Fig 07 — Supply vs demand share by statedf = q("""WITH cust AS (SELECT customer_state AS state, COUNT(DISTINCT customer_unique_id) AS customers              FROM `olist.v_orders_enriched` GROUP BY state),     sell AS (SELECT UPPER(TRIM(seller_state)) AS state, COUNT(*) AS sellers              FROM `olist.sellers` GROUP BY state)SELECT COALESCE(c.state,s.state) AS state,       IFNULL(c.customers,0) AS customers, IFNULL(s.sellers,0) AS sellers,       ROUND(100.0*IFNULL(c.customers,0)/SUM(IFNULL(c.customers,0)) OVER (),2) AS pct_demand,       ROUND(100.0*IFNULL(s.sellers,0)/SUM(IFNULL(s.sellers,0)) OVER (),2) AS pct_supplyFROM cust c FULL OUTER JOIN sell s ON c.state=s.stateORDER BY customers DESC""")d = df.head(12).iloc[::-1]y = np.arange(len(d)); h = 0.38fig, ax = plt.subplots(figsize=(6, 4))ax.barh(y - h/2, d.pct_demand, h, label="Demand (customers)", color=C_MAIN)ax.barh(y + h/2, d.pct_supply, h, label="Supply (sellers)",  color=C_ACCENT)ax.set_yticks(y); ax.set_yticklabels(d.state)ax.set_xlabel("Share of total (%)")ax.set_title("Supply is far more concentrated in São Paulo than demand")ax.legend(loc="lower right")save(fig, "fig_07_supply_demand_state"); plt.show()

In [ ]:
# Fig 08 — State scatter: delivery time vs satisfaction (bubble = volume)df = q("""SELECT customer_state, COUNT(*) AS items,       ROUND(AVG(delivery_days),1) AS avg_delivery_days,       ROUND(AVG(review_score),3)  AS avg_review_score,       ROUND(AVG(distance_km),0)   AS avg_distance_kmFROM `olist.v_fact_order_items`WHERE order_status='delivered'GROUP BY customer_state HAVING items >= 100""")fig, ax = plt.subplots(figsize=(6, 4))sc = ax.scatter(df.avg_delivery_days, df.avg_review_score,                s=np.sqrt(df["items"])*1.5, c=df.avg_distance_km,                cmap="plasma", alpha=0.85, edgecolor="white", linewidth=0.5)for _, r in df.iterrows():    ax.annotate(r.customer_state, (r.avg_delivery_days, r.avg_review_score),                fontsize=6.5, ha="center", va="center", color="white", weight="bold")ax.set_xlabel("Average delivery time (days)")ax.set_ylabel("Average review score")ax.set_title("States that wait longer rate lower (bubble = volume, colour = distance)")fig.colorbar(sc, ax=ax, label="Avg distance (km)", fraction=0.04, pad=0.02)save(fig, "fig_08_state_scatter"); plt.show()

---## 4 · Chapter 4 — Retention: the 3.1% problem*Verified: 96,096 unique people, 93,099 ordered exactly once. Using `customer_id`instead of `customer_unique_id` reports a false 0.00% repeat rate.*

In [ ]:
# Fig 09 — Orders per person (the fat "1" bar)df = q("""WITH opp AS (SELECT customer_unique_id, COUNT(DISTINCT order_id) AS n             FROM `olist.v_orders_enriched` GROUP BY customer_unique_id)SELECT n AS order_count, COUNT(*) AS customers FROM opp GROUP BY n ORDER BY n""")d = df[df.order_count <= 7]fig, ax = plt.subplots(figsize=(6, 3.2))cols = [C_WARN] + [C_MAIN]*(len(d)-1)b = ax.bar(d.order_count.astype(str), d.customers, color=cols)ax.set_yscale("log")ax.set_xlabel("Orders placed"); ax.set_ylabel("Customers (log scale)")ax.set_title("96.9% of customers order exactly once")for bar, v in zip(b, d.customers):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.15,            f"{v:,}", ha="center", fontsize=7)save(fig, "fig_09_orders_per_person"); plt.show()

In [ ]:
# Fig 10 — Cohort retention triangledf = q("""WITH fp AS (SELECT customer_unique_id, DATE_TRUNC(DATE(MIN(purchase_ts)),MONTH) AS cohort_month            FROM `olist.v_orders_enriched` GROUP BY customer_unique_id),     act AS (SELECT e.customer_unique_id, fp.cohort_month,                    DATE_DIFF(DATE_TRUNC(DATE(e.purchase_ts),MONTH), fp.cohort_month, MONTH) AS off             FROM `olist.v_orders_enriched` e JOIN fp USING (customer_unique_id))SELECT cohort_month, off AS month_offset, COUNT(DISTINCT customer_unique_id) AS activeFROM act GROUP BY cohort_month, month_offset ORDER BY cohort_month, month_offset""")df["cohort_month"] = pd.to_datetime(df.cohort_month)df = df[(df.cohort_month >= "2017-01-01") & (df.cohort_month <= "2018-06-01")]df = df[(df.month_offset >= 0) & (df.month_offset <= 11)]piv  = df.pivot(index="cohort_month", columns="month_offset", values="active")rate = piv.div(piv[0], axis=0) * 100fig, ax = plt.subplots(figsize=(7, 4))im = ax.imshow(rate.values, aspect="auto", cmap="Blues", vmin=0, vmax=3)ax.set_yticks(range(len(rate))); ax.set_yticklabels(rate.index.strftime("%b %Y"), fontsize=7)ax.set_xticks(range(rate.shape[1])); ax.set_xticklabels(rate.columns)ax.set_xlabel("Months since first purchase"); ax.set_ylabel("Cohort")ax.set_title("Retention collapses immediately — the curve never recovers")for i in range(rate.shape[0]):    for j in range(1, rate.shape[1]):        v = rate.values[i, j]        if not np.isnan(v):            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=5.5)ax.grid(False)fig.colorbar(im, ax=ax, label="Retention (%)", fraction=0.03, pad=0.02)save(fig, "fig_10_cohort_triangle"); plt.show()

In [ ]:
# Fig 11 — Time to second purchasedf = q("""WITH r AS (SELECT customer_unique_id, purchase_ts,                  ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY purchase_ts) AS seq           FROM `olist.v_orders_enriched`),     g AS (SELECT customer_unique_id,                  DATE_DIFF(DATE(MAX(IF(seq=2,purchase_ts,NULL))),                            DATE(MAX(IF(seq=1,purchase_ts,NULL))), DAY) AS days_gap           FROM r WHERE seq<=2 GROUP BY customer_unique_id)SELECT days_gap FROM g WHERE days_gap IS NOT NULL""")fig, ax = plt.subplots(figsize=(6, 3))ax.hist(df.days_gap, bins=60, color=C_MAIN, edgecolor="white", linewidth=0.3)med = df.days_gap.median()ax.axvline(med, color=C_WARN, ls="--", lw=1.2, label=f"Median = {med:.0f} days")ax.set_xlabel("Days between 1st and 2nd order"); ax.set_ylabel("Customers")ax.set_title("Among the 3% who return, most take months to come back")ax.legend()save(fig, "fig_11_time_to_second"); plt.show()

---## 6 · Chapter 6 — Payments: the Brazil-specific chapter*Parcelamento (installments) is a genuine behavioural lever most generic analyses skip.*

In [ ]:
# Fig 12 — Payment mix and installments vs AOVmix = q("""SELECT primary_payment_type, COUNT(*) AS orders, ROUND(AVG(order_payment_total),2) AS aovFROM `olist.v_payments_order` GROUP BY primary_payment_type ORDER BY orders DESC""")ins = q("""SELECT max_installments AS installments, COUNT(*) AS orders,       ROUND(AVG(order_payment_total),2) AS aovFROM `olist.v_payments_order` GROUP BY installments ORDER BY installments""")ins = ins[(ins.installments >= 1) & (ins.installments <= 12)]fig, axes = plt.subplots(1, 2, figsize=(7.2, 3))axes[0].bar(mix.primary_payment_type, mix.orders, color=C_MAIN)axes[0].set_title("Payment method mix"); axes[0].set_ylabel("Orders")axes[0].tick_params(axis="x", rotation=30)axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))a = axes[1]; a2 = a.twinx()a.bar(ins.installments, ins.orders, color=C_MUTED, alpha=0.55, label="Orders")a2.plot(ins.installments, ins.aov, marker="o", ms=3.5, color=C_WARN, label="AOV")a.set_xlabel("Installments"); a.set_ylabel("Orders"); a2.set_ylabel("AOV (R$)")a.set_title("Bigger baskets are financed over more installments")a.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))a2.grid(False)save(fig, "fig_12_payments"); plt.show()

---## 7 · Chapter 7 — Delivery, freight & unit economics*Where the time goes, and where fulfilment stops paying for itself.*

In [ ]:
# Fig 13 — Delay decomposition: which leg eats the time?df = q("""SELECT ROUND(AVG(DATE_DIFF(DATE(approved_ts), DATE(purchase_ts), DAY)),2) AS approve, ROUND(AVG(DATE_DIFF(DATE(carrier_ts), DATE(approved_ts), DAY)),2)  AS seller_to_carrier, ROUND(AVG(DATE_DIFF(DATE(delivered_ts), DATE(carrier_ts), DAY)),2) AS carrier_to_customerFROM `olist.v_orders_enriched`WHERE order_status='delivered' AND approved_ts IS NOT NULL  AND carrier_ts IS NOT NULL AND delivered_ts IS NOT NULL""")legs = ["Purchase→Approved", "Approved→Carrier", "Carrier→Customer"]vals = [df.approve[0], df.seller_to_carrier[0], df.carrier_to_customer[0]]fig, ax = plt.subplots(figsize=(6, 2.2))left = 0for lab, v, c in zip(legs, vals, [C_MUTED, C_ACCENT, C_MAIN]):    ax.barh([0], [v], left=left, color=c, label=f"{lab} ({v:.1f}d)", height=0.5)    if v > 0.6:        ax.text(left + v/2, 0, f"{v:.1f}", ha="center", va="center",                color="white", fontsize=8, weight="bold")    left += vax.set_yticks([]); ax.set_xlabel("Days")ax.set_title(f"Where the {sum(vals):.1f} days go — the carrier leg dominates")ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)save(fig, "fig_13_delay_decomposition"); plt.show()

In [ ]:
# Fig 14 — Distance drives both delivery time and freightdf = q("""SELECT CASE WHEN distance_km<50 THEN '0-50' WHEN distance_km<200 THEN '50-200'            WHEN distance_km<500 THEN '200-500' WHEN distance_km<1000 THEN '500-1000'            ELSE '1000+' END AS bucket,       MIN(distance_km) AS srt, COUNT(*) AS items,       ROUND(AVG(delivery_days),1) AS avg_delivery_days,       ROUND(AVG(freight_value),2) AS avg_freight,       ROUND(AVG(review_score),3)  AS avg_review_scoreFROM `olist.v_fact_order_items`WHERE order_status='delivered' AND distance_km IS NOT NULLGROUP BY bucket ORDER BY srt""")fig, ax = plt.subplots(figsize=(6.5, 3.2))x = np.arange(len(df)); w = 0.38ax.bar(x - w/2, df.avg_delivery_days, w, color=C_MAIN, label="Delivery days")ax.bar(x + w/2, df.avg_freight, w, color=C_ACCENT, label="Freight (R$)")ax2 = ax.twinx()ax2.plot(x, df.avg_review_score, marker="o", ms=4, color=C_WARN, label="Review score")ax2.set_ylabel("Average review score"); ax2.grid(False)ax.set_xticks(x); ax.set_xticklabels(df.bucket)ax.set_xlabel("Seller→customer distance (km)")ax.set_title("Distance raises cost and wait time, and drags satisfaction down")h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()ax.legend(h1+h2, l1+l2, loc="upper left", fontsize=7)save(fig, "fig_14_distance_effects"); plt.show()

In [ ]:
# Fig 15 — Freight economics: cheap items are the ones that bleeddf = q("""SELECT CASE WHEN price<30 THEN 'R$0-30' WHEN price<60 THEN 'R$30-60'            WHEN price<120 THEN 'R$60-120' WHEN price<300 THEN 'R$120-300'            ELSE 'R$300+' END AS band,       MIN(price) AS srt, COUNT(*) AS items,       ROUND(AVG(price),2) AS avg_price, ROUND(AVG(freight_value),2) AS avg_freight,       ROUND(100.0*AVG(freight_value/NULLIF(price,0)),1) AS freight_pctFROM `olist.v_fact_order_items` GROUP BY band ORDER BY srt""")fig, ax = plt.subplots(figsize=(6.5, 3.2))x = np.arange(len(df))b = ax.bar(x, df.freight_pct, color=[C_WARN if v > 50 else C_MAIN for v in df.freight_pct])ax.axhline(100, color="black", ls=":", lw=0.9)ax.text(len(df)-0.5, 103, "freight = item price", fontsize=7, ha="right")ax.set_xticks(x); ax.set_xticklabels(df.band)ax.set_ylabel("Freight as % of item price"); ax.set_xlabel("Item price band")ax.set_title("On sub-R\\$30 items, shipping costs nearly as much as the product")barlabels(ax, b, "{:.0f}%")save(fig, "fig_15_freight_economics"); plt.show()

---## 8 · Chapter 8 — Sellers: concentration and quality risk

In [ ]:
# Fig 16 — Seller Pareto curvedf = q("""WITH s AS (SELECT seller_id, SUM(price) AS revenue FROM `olist.v_fact_order_items`           GROUP BY seller_id)SELECT ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rank, revenue,       SUM(revenue) OVER (ORDER BY revenue DESC) AS cum_rev,       SUM(revenue) OVER () AS total_rev, COUNT(*) OVER () AS n_sellersFROM s ORDER BY rank""")df["pct_sellers"] = 100*df["rank"]/df.n_sellersdf["pct_revenue"] = 100*df.cum_rev/df.total_revfig, ax = plt.subplots(figsize=(5.6, 3.4))ax.plot(df.pct_sellers, df.pct_revenue, color=C_MAIN, lw=1.6)ax.plot([0,100],[0,100], ls="--", color="grey", lw=0.8, label="Perfect equality")p20 = df.loc[(df.pct_sellers-20).abs().idxmin(), "pct_revenue"]ax.axvline(20, color=C_WARN, ls=":", lw=1)ax.annotate(f"Top 20% of sellers\n= {p20:.0f}% of revenue", xy=(20, p20),            xytext=(28, p20-22), fontsize=8, color=C_WARN,            arrowprops=dict(arrowstyle="->", color=C_WARN, lw=0.8))ax.set_xlabel("Cumulative % of sellers"); ax.set_ylabel("Cumulative % of revenue")ax.set_title("Revenue is highly concentrated among few sellers")ax.legend(loc="lower right", fontsize=7)save(fig, "fig_16_seller_pareto"); plt.show()

In [ ]:
# Fig 17 — Do big sellers deliver good experiences?df = q("""WITH s AS (SELECT seller_id, COUNT(*) AS items, SUM(price) AS revenue,                  AVG(review_score) AS avg_score, AVG(delivery_days) AS avg_days,                  AVG(IF(lateness_days>0,1,0)) AS late_rate           FROM `olist.v_fact_order_items` WHERE order_status='delivered'           GROUP BY seller_id HAVING items>=20),     t AS (SELECT s.*, NTILE(4) OVER (ORDER BY revenue DESC) AS quartile FROM s)SELECT quartile, COUNT(*) AS sellers, ROUND(AVG(avg_score),3) AS avg_review_score,       ROUND(AVG(avg_days),1) AS avg_delivery_days, ROUND(100.0*AVG(late_rate),1) AS pct_lateFROM t GROUP BY quartile ORDER BY quartile""")fig, ax = plt.subplots(figsize=(6, 3.2))x = np.arange(len(df))b = ax.bar(x, df.avg_review_score, color=C_MAIN, width=0.5)ax2 = ax.twinx()ax2.plot(x, df.pct_late, marker="s", ms=4, color=C_WARN, label="% late")ax2.set_ylabel("% delivered late"); ax2.grid(False)ax.set_xticks(x); ax.set_xticklabels([f"Q{i}" for i in df.quartile])ax.set_xlabel("Seller revenue quartile (Q1 = largest)")ax.set_ylabel("Average review score"); ax.set_ylim(3.5, 4.5)ax.set_title("Seller quality by revenue tier — is the top tier a risk?")barlabels(ax, b, "{:.2f}", dy=0.005)ax2.legend(loc="lower right", fontsize=7)save(fig, "fig_17_seller_quality_tiers"); plt.show()

---## 9 · Chapter 9 — Reviews & satisfaction**Fig 19 is the centrepiece of the entire report.**

In [ ]:
# Fig 18 — Review score distributiondf = q("""SELECT review_score, COUNT(*) AS reviews,       ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pctFROM `olist.v_reviews_clean` GROUP BY review_score ORDER BY review_score""")fig, ax = plt.subplots(figsize=(5.4, 3))cols = [C_WARN, C_WARN, C_MUTED, C_MAIN, C_MAIN]b = ax.bar(df.review_score.astype(str), df.reviews, color=cols)ax.set_xlabel("Review score"); ax.set_ylabel("Reviews")ax.set_title("Ratings are polarised: mostly 5s, with a hard core of 1s")ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))for bar, p in zip(b, df.pct):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,            f"{p:.1f}%", ha="center", fontsize=7)save(fig, "fig_18_review_distribution"); plt.show()

In [ ]:
# Fig 19 *** CENTREPIECE *** Lateness destroys satisfactiondf = q("""WITH base AS (  SELECT DATE_DIFF(DATE(o.delivered_ts), DATE(o.estimated_ts), DAY) AS gap, r.review_score  FROM `olist.v_orders_enriched` o JOIN `olist.v_reviews_clean` r USING (order_id)  WHERE o.order_status='delivered' AND o.delivered_ts IS NOT NULL AND o.estimated_ts IS NOT NULL)SELECT CASE WHEN gap<=-1 THEN 'Early' WHEN gap=0 THEN 'On time'            WHEN gap BETWEEN 1 AND 3 THEN '1-3d late'            WHEN gap BETWEEN 4 AND 7 THEN '4-7d late' ELSE '8d+ late' END AS bucket,       MIN(gap) AS srt, COUNT(*) AS orders,       ROUND(AVG(review_score),3) AS avg_score,       ROUND(100.0*COUNTIF(review_score<=2)/COUNT(*),1) AS pct_1_2_starFROM base GROUP BY bucket ORDER BY srt""")fig, ax = plt.subplots(figsize=(6.5, 3.6))x = np.arange(len(df))b = ax.bar(x, df.avg_score, color=C_MAIN, width=0.5, label="Avg review score")ax2 = ax.twinx()ax2.plot(x, df.pct_1_2_star, marker="o", ms=5, color=C_WARN, lw=1.8, label="% 1–2 star")ax2.set_ylabel("% of orders rated 1–2 star", color=C_WARN)ax2.tick_params(axis="y", colors=C_WARN); ax2.grid(False)ax.set_xticks(x); ax.set_xticklabels(df.bucket)ax.set_ylabel("Average review score"); ax.set_ylim(0, 5)ax.set_xlabel("Delivery vs promised date")ax.set_title("Late delivery is the single strongest driver of bad reviews", weight="bold")barlabels(ax, b, "{:.2f}", dy=0.02)h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()ax.legend(h1+h2, l1+l2, loc="lower left", fontsize=7)save(fig, "fig_19_lateness_vs_review"); plt.show()

In [ ]:
# Fig 20 — Estimate padding: Olist sandbags its own promisesdf = q("""SELECT DATE_DIFF(DATE(estimated_ts), DATE(delivered_ts), DAY) AS days_earlyFROM `olist.v_orders_enriched`WHERE order_status='delivered' AND delivered_ts IS NOT NULL AND estimated_ts IS NOT NULL""")d = df[(df.days_early > -30) & (df.days_early < 60)]fig, ax = plt.subplots(figsize=(6.2, 3))ax.hist(d.days_early, bins=80, color=C_MAIN, edgecolor="white", linewidth=0.2)ax.axvline(0, color="black", lw=1)med = d.days_early.median()ax.axvline(med, color=C_WARN, ls="--", lw=1.2, label=f"Median = {med:.0f} days early")ax.text(-16, ax.get_ylim()[1]*0.75, "LATE", color=C_WARN, fontsize=8, weight="bold")ax.set_xlabel("Days earlier than the promised date"); ax.set_ylabel("Orders")ax.set_title("Delivery estimates are heavily padded — most orders arrive well early")ax.legend(fontsize=7)save(fig, "fig_20_estimate_padding"); plt.show()

---## 10 · Chapter 10 — Synthesis: the causal chain*The two figures that close the argument.*

In [ ]:
# Fig 21 — THE CHAIN: distance -> delivery -> lateness -> score -> repeatdf = q("""WITH il AS (SELECT customer_unique_id, order_id, distance_km, delivery_days,                   lateness_days, review_score            FROM `olist.v_fact_order_items`            WHERE order_status='delivered' AND distance_km IS NOT NULL),     ol AS (SELECT customer_unique_id, order_id, AVG(distance_km) AS distance_km,                   AVG(delivery_days) AS delivery_days, AVG(lateness_days) AS lateness_days,                   AVG(review_score) AS review_score            FROM il GROUP BY customer_unique_id, order_id),     co AS (SELECT customer_unique_id, COUNT(DISTINCT order_id) AS n            FROM `olist.v_orders_enriched` GROUP BY customer_unique_id)SELECT CASE WHEN o.distance_km<200 THEN '0-200' WHEN o.distance_km<500 THEN '200-500'            WHEN o.distance_km<1000 THEN '500-1000' ELSE '1000+' END AS bucket,       MIN(o.distance_km) AS srt, COUNT(*) AS orders,       ROUND(AVG(o.delivery_days),1) AS avg_delivery_days,       ROUND(100.0*AVG(IF(o.lateness_days>0,1,0)),1) AS pct_late,       ROUND(AVG(o.review_score),3) AS avg_review_score,       ROUND(100.0*COUNTIF(c.n>1)/COUNT(*),2) AS repeat_rate_pctFROM ol o JOIN co c USING (customer_unique_id)GROUP BY bucket ORDER BY srt""")metrics = [("avg_delivery_days","Delivery days",C_MAIN),           ("pct_late","% late",C_MUTED),           ("avg_review_score","Review score",C_ACCENT),           ("repeat_rate_pct","Repeat rate (%)",C_WARN)]fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))for ax, (col, lab, c) in zip(axes, metrics):    ax.bar(df.bucket, df[col], color=c, width=0.6)    ax.set_title(lab, fontsize=9)    ax.tick_params(axis="x", rotation=45, labelsize=7)    for i, v in enumerate(df[col]):        ax.text(i, v, f"{v:g}", ha="center", va="bottom", fontsize=6.5)axes[0].set_ylabel("Value")fig.suptitle("The chain: greater distance → slower, later delivery → worse reviews → fewer returns",             y=1.06, fontsize=10, weight="bold")fig.text(0.5, -0.10, "Seller→customer distance (km)", ha="center", fontsize=9)save(fig, "fig_21_causal_chain"); plt.show()display(df)

In [ ]:
# Fig 22 — Did a bad first order kill the relationship?df = q("""WITH fo0 AS (SELECT customer_unique_id, order_id, purchase_ts,                    ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY purchase_ts) AS seq             FROM `olist.v_orders_enriched`),     fo AS (SELECT * FROM fo0 WHERE seq=1),     tot AS (SELECT customer_unique_id, COUNT(DISTINCT order_id) AS n             FROM `olist.v_orders_enriched` GROUP BY customer_unique_id),     en AS (SELECT fo.customer_unique_id, r.review_score,                   DATE_DIFF(DATE(e.delivered_ts), DATE(e.estimated_ts), DAY) AS lateness,                   t.n            FROM fo            JOIN `olist.v_orders_enriched` e ON fo.order_id=e.order_id            JOIN tot t ON fo.customer_unique_id=t.customer_unique_id            LEFT JOIN `olist.v_reviews_clean` r ON fo.order_id=r.order_id            WHERE e.order_status='delivered' AND e.delivered_ts IS NOT NULL)SELECT review_score AS first_score, COUNT(*) AS customers,       ROUND(100.0*COUNTIF(n>1)/COUNT(*),2) AS return_rate_pctFROM en WHERE review_score IS NOT NULL GROUP BY first_score ORDER BY first_score""")fig, ax = plt.subplots(figsize=(5.6, 3.2))cols = [C_WARN, C_WARN, C_MUTED, C_MAIN, C_MAIN]b = ax.bar(df.first_score.astype(str), df.return_rate_pct, color=cols, width=0.55)ax.set_xlabel("Review score on FIRST order")ax.set_ylabel("% who ever ordered again")ax.set_title("Does a bad first impression end the relationship?", weight="bold")barlabels(ax, b, "{:.2f}%", dy=0.02)save(fig, "fig_22_first_order_retention"); plt.show()display(df)

---## 11 · Export figures

In [ ]:
import shutilshutil.make_archive("olist_figures", "zip", FIGDIR)print(sorted(os.listdir(FIGDIR)))from google.colab import filesfiles.download("olist_figures.zip")

### Using these in LaTeX```latex\usepackage{graphicx}\begin{figure}[htbp]  \centering  \includegraphics[width=\linewidth]{figures/fig_19_lateness_vs_review.pdf}  \caption{Late delivery is the strongest single driver of poor review scores.}  \label{fig:lateness}\end{figure}```Set `USE_LATEX = True` and re-run before final export so the figure fonts match your document.